# CS322 - Gold Trend Prediction with a Deep (Stacked + Dense) LSTM

**Goal:** predict the next trading day's gold close (USD/oz, COMEX futures) from the previous 60 days.

**Model:** 3 stacked LSTM layers (128 units) followed by a dense head 128 → 64 → 32 → 1.

**Pipeline:**
1. Load data
2. Build features
3. Chronological split: train 2016–2023 / validation 2024 / test 2025–Jan 2026
4. Train with MSE (least-squares) loss and early stopping
5. Evaluate against baselines
6. Forecast forward and compare with the real Nov 2026 price

Requirements: `pip install torch numpy pandas matplotlib`

In [ ]:
import os, random, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)

CSV = 'gold_historical_data.csv'       # keep the notebook in the same folder as the CSV
OUT = 'lstm_deep_outputs'
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

## 1. Load data

In [ ]:
df = pd.read_csv(CSV, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
print(df.shape, df['Date'].min().date(), '->', df['Date'].max().date())
df.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df['Date'], df['Close'], color='black', lw=1)
ax.set_title('CS322 - Gold futures daily close 2016-2026'); ax.set_ylabel('USD / oz')
plt.show()

## 2. Features

Raw prices are **not** used as inputs. Training data tops out near \$2,100 (2023), but the test period reaches \$5,300, and a price-scaled network cannot output levels it has never seen. Every feature is therefore scale-free:

| feature | formula | meaning |
|---|---|---|
| `ret` | ln(Close_t / Close_t-1) | daily log-return |
| `hl` | ln(High_t / Low_t) | intraday range (volatility) |
| `oc` | ln(Close_t / Open_t) | intraday direction |

**Target:** tomorrow's log-return r. The predicted price is Close_t · e^r̂.

In [ ]:
close = df['Close'].to_numpy(float)
feat = pd.DataFrame({
    'ret': np.log(df['Close']).diff(),
    'hl':  np.log(df['High'] / df['Low']),
    'oc':  np.log(df['Close'] / df['Open']),
})
feat = feat.replace([np.inf, -np.inf], np.nan).fillna(0.0)
target = np.log(df['Close']).diff().shift(-1)      # return from day t to t+1
feat.describe().T

## 3. Windows and chronological split (standardised with **train** statistics only)

In [ ]:
WINDOW = 60
TRAIN_END, VALID_END = pd.Timestamp('2023-12-31'), pd.Timestamp('2024-12-31')

dates = df['Date']
F = feat.to_numpy(np.float32)
tr_rows = dates <= TRAIN_END
f_mu, f_sd = F[tr_rows].mean(0), F[tr_rows].std(0)
y_mu, y_sd = np.nanmean(target[tr_rows]), np.nanstd(target[tr_rows])
Fz = (F - f_mu) / f_sd
yz = ((target - y_mu) / y_sd).to_numpy(np.float32)

X, Y, t_idx = [], [], []                  # sample uses days t-59..t, predicts close at t+1
for t in range(WINDOW, len(df) - 1):
    X.append(Fz[t - WINDOW + 1:t + 1]); Y.append(yz[t]); t_idx.append(t)
X, Y, t_idx = np.stack(X), np.array(Y), np.array(t_idx)
pred_dates = dates.iloc[t_idx + 1].to_numpy()

tr = pred_dates <= np.datetime64(TRAIN_END)
va = (pred_dates > np.datetime64(TRAIN_END)) & (pred_dates <= np.datetime64(VALID_END))
te = pred_dates > np.datetime64(VALID_END)
print('X shape', X.shape, '| train', tr.sum(), 'valid', va.sum(), 'test', te.sum())

## 4. Deep LSTM model (stacked LSTM + dense head)

In [ ]:
class DeepLSTM(nn.Module):
    def __init__(self, n_feat=3, hidden=128, layers=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, num_layers=layers, batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(hidden)
        self.dense = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),     nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),      nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)                     # (batch, 60, hidden)
        return self.dense(self.norm(out[:, -1])).squeeze(-1)

model = DeepLSTM().to(DEVICE)
print(model)
print('trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. Training (MSE loss, AdamW, LR scheduling, early stopping on validation)

In [ ]:
EPOCHS, BATCH, PATIENCE = 200, 64, 25
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=8)
loss_fn = nn.MSELoss()

Xt = torch.tensor(X, device=DEVICE); Yt = torch.tensor(Y, device=DEVICE)
idx_tr = np.where(tr)[0]

def eval_loss(mask):
    model.eval()
    with torch.no_grad():
        return loss_fn(model(Xt[mask]), Yt[mask]).item()

hist, best, best_state, wait, t0 = [], np.inf, None, 0, time.time()
for ep in range(1, EPOCHS + 1):
    model.train(); np.random.shuffle(idx_tr)
    for b in range(0, len(idx_tr), BATCH):
        bi = idx_tr[b:b + BATCH]
        opt.zero_grad()
        loss = loss_fn(model(Xt[bi]), Yt[bi])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    tl, vl = eval_loss(tr), eval_loss(va)
    sched.step(vl); hist.append((tl, vl))
    if vl < best - 1e-5:
        best, wait, best_state = vl, 0, copy.deepcopy(model.state_dict())
    else:
        wait += 1
    if ep % 10 == 0 or wait == 0:
        print(f'epoch {ep:3d}  train {tl:.4f}  valid {vl:.4f}  lr {opt.param_groups[0]["lr"]:.1e}')
    if wait >= PATIENCE:
        print(f'early stop at epoch {ep}'); break
model.load_state_dict(best_state)
print(f'best valid MSE {best:.4f} | {time.time()-t0:.0f}s')

In [ ]:
h = np.array(hist)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(h[:, 0], label='train'); ax.plot(h[:, 1], label='validation')
ax.set_xlabel('epoch'); ax.set_ylabel('MSE (standardised)')
ax.set_title('CS322 - Deep LSTM training curve'); ax.legend()
fig.savefig(f'{OUT}/fig_training.png', bbox_inches='tight'); plt.show()

## 6. Evaluation against baselines

- **Naive (random walk):** Close_t+1 = Close_t
- **Least-squares lag-1:** Close_t+1 = a + b·Close_t, fitted on train by solving the normal equations (AᵀA)β = Aᵀy

In [ ]:
model.eval()
with torch.no_grad():
    r_hat = model(Xt).cpu().numpy() * y_sd + y_mu
prev = close[t_idx]; actual = close[t_idx + 1]
pred_lstm = prev * np.exp(r_hat)
A = np.column_stack([np.ones(tr.sum()), prev[tr]])
beta = np.linalg.solve(A.T @ A, A.T @ actual[tr])
pred_ls = beta[0] + beta[1] * prev

def metrics(y, p, pv):
    e = y - p; sse = np.sum(e**2)
    return dict(RMSE=np.sqrt(np.mean(e**2)), MAE=np.mean(np.abs(e)),
                MAPE=np.mean(np.abs(e / y)) * 100, R2=1 - sse / np.sum((y - y.mean())**2),
                SSE=sse, DirAcc=np.mean(np.sign(p - pv) == np.sign(y - pv)) * 100)

rows = []
for split, m in [('valid 2024', va), ('test 2025-Jan26', te)]:
    for name, p in [('Deep LSTM', pred_lstm), ('Naive', prev), ('LS lag-1', pred_ls)]:
        rows.append({'split': split, 'model': name, **metrics(actual[m], p[m], prev[m])})
res = pd.DataFrame(rows); res.to_csv(f'{OUT}/metrics.csv', index=False)
print('Up-day share in test: %.1f%%  | LSTM predicts up: %.1f%%' % (
    (actual[te] > prev[te]).mean() * 100, (pred_lstm[te] > prev[te]).mean() * 100))
res.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(pred_dates[te], actual[te], color='black', lw=1.6, label='Actual')
ax.plot(pred_dates[te], pred_lstm[te], color='tab:orange', lw=1.1, label='Deep LSTM (next-day)')
ax.plot(pred_dates[te], pred_ls[te], color='tab:blue', lw=0.9, ls='--', label='LS lag-1')
ax.set_ylabel('USD / oz'); ax.legend()
ax.set_title('CS322 - Deep LSTM next-day prediction on unseen test data')
fig.savefig(f'{OUT}/fig_test.png', bbox_inches='tight'); plt.show()

pd.DataFrame({'Date': pred_dates[te], 'Actual': actual[te], 'DeepLSTM': pred_lstm[te],
              'Naive': prev[te], 'LS_lag1': pred_ls[te]}).to_csv(f'{OUT}/test_predictions.csv', index=False)

## 7. Forecast: next trading day, then recursively to 10 Nov 2026

For recursive steps, future intraday features (`hl`, `oc`) are unknown, so they are held at their recent 60-day average. This is a limitation of multi-step forecasting worth stating in the report.

In [ ]:
def predict_ret(window_z):
    with torch.no_grad():
        x = torch.tensor(window_z[None], dtype=torch.float32, device=DEVICE)
        return model(x).item() * y_sd + y_mu

win = Fz[-WINDOW:].copy()
r1 = predict_ret(win)
print(f'Last close {dates.iloc[-1].date()}: {close[-1]:.2f}')
print(f'Next-day forecast (2026-02-02): {close[-1]*np.exp(r1):.2f}')

fdates = pd.bdate_range(dates.iloc[-1] + pd.Timedelta(days=1), '2026-11-10')
hl_oc_mean = Fz[-WINDOW:, 1:].mean(0)
c, path = close[-1], []
for _ in fdates:
    r = predict_ret(win); c *= np.exp(r); path.append(c)
    new = np.r_[(r - f_mu[0]) / f_sd[0], hl_oc_mean].astype(np.float32)
    win = np.vstack([win[1:], new])
fc = pd.DataFrame({'Date': fdates, 'DeepLSTM_forecast': path})
fc.to_csv(f'{OUT}/forecast_to_nov2026.csv', index=False)

ACTUAL_NOV9 = 4123   # Fortune, 9 Nov 2026 (~$4,123/oz)
print(f'Forecast for 2026-11-10: {path[-1]:.2f} | actual ~{ACTUAL_NOV9} | error {(path[-1]/ACTUAL_NOV9-1)*100:+.1f}%')

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(dates, close, color='black', lw=1, label='Historical')
ax.plot(fc['Date'], fc['DeepLSTM_forecast'], color='tab:orange', lw=1.5, label='Deep LSTM recursive forecast')
ax.scatter([pd.Timestamp('2026-11-09')], [ACTUAL_NOV9], color='tab:red', zorder=5, label='Actual 9 Nov 2026')
ax.axvline(dates.iloc[-1], color='grey', ls=':'); ax.set_ylabel('USD / oz'); ax.legend(loc='upper left')
ax.set_title('CS322 - Deep LSTM forecast Feb - Nov 2026')
fig.savefig(f'{OUT}/fig_forecast.png', bbox_inches='tight'); plt.show()

## 8. Save the model

In [ ]:
torch.save({'state_dict': model.state_dict(), 'f_mu': f_mu, 'f_sd': f_sd,
            'y_mu': y_mu, 'y_sd': y_sd, 'window': WINDOW}, f'{OUT}/deep_lstm.pt')
print('saved to', OUT)